In [2]:
# Fear Conditioningのデータを読み込み

from pathlib import Path  # noqa: F811

import numpy as np  # noqa: F811
import pandas as pd

# 自動でエクセルファイルを読み込む

folder = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\FC"

files = sorted(Path(folder).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート

if not files:
    print(f"{folder} の中に .xls ファイルはありません")
else:
    dfs = [] # 空のリストを作成して、各ファイルのデータフレームを格納
    for file in files: # ファイルごとにループ
        df = (
        pd.read_excel(file) 
        .iloc[2:8, [4]]  # 2行目から7行目まで、4列目を抽出
        .assign(
            No = lambda df: Path(file).stem, # No列にpathからファイル名を抽出して追加
            Group = lambda df: np.select( # Group列に条件に応じた値を追加
                condlist=[ # 条件のリスト：No列に各文字列が含まれているか
                df['No'].str.contains('SED'),
                df['No'].str.contains('LIE'),
                df['No'].str.contains('MOE')
            ],
            choicelist=['SED', 'LIE', 'MOE'], # 条件にマッチしたときに入れる値のリスト
            default='Other' # どれにも当てはまらない場合のデフォルト値
            ),
            Time  = lambda df: list(range(1, len(df) + 1)), # Time列に1から行数までの連番を追加
        )
        .assign(Freezing = lambda df: df['Interval.3'] / 60 * 100 ) # Freezing Time (%) を計算し列に追加
        .iloc[:, 1:5]
        )
        dfs.append(df) # データフレームをリストに追加

    dataFC = pd.concat(dfs, ignore_index=True) # リスト内のデータフレームを縦に結合して1つのデータフレームにする

    print(f"{len(files)} 件の .xls ファイルを読み込みました") # 読み込んだファイル数を表示
    print(dataFC) # データフレームの内容を表示

24 件の .xls ファイルを読み込みました
           No Group  Time   Freezing
0    100FCMOE   MOE     1        0.0
1    100FCMOE   MOE     2        0.0
2    100FCMOE   MOE     3        0.0
3    100FCMOE   MOE     4       47.6
4    100FCMOE   MOE     5       43.5
..        ...   ...   ...        ...
139   99FCMOE   MOE     2        0.0
140   99FCMOE   MOE     3        0.0
141   99FCMOE   MOE     4  14.566667
142   99FCMOE   MOE     5       60.8
143   99FCMOE   MOE     6  71.966667

[144 rows x 4 columns]


In [3]:
# Fear Extinctionのデータを読み込み

from pandas.core.arrays import categorical
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# 旧形式 .xls の OLE2 警告を抑制
# warnings.filterwarnings("ignore", message=".*OLE2 inconsistency.*")

def infer_group(name: str) -> str: # ファイル名からグループを推測する関数
    if "SED" in name:
        return "SED"
    elif "LIE" in name:
        return "LIE"
    elif "MOE" in name:
        return "MOE"
    return "Other"

def read_extinction_per3(files):   # 3分ごとのデータを読み込む関数
    rows = [] # 空のリストを作成して、各ファイルのデータを格納

    for file in files:
        # 旧 .xls では pandas で警告が出ることがあるので抑制
        # with warnings.catch_warnings():
            # warnings.filterwarnings("ignore", message=".*OLE2 inconsistency.*")
        df = pd.read_excel(file)

        col5 = pd.to_numeric(df.iloc[:, 4], errors="coerce")  # 5列目（E列）
        bins = [ # 3分毎のbinの範囲とラベル
            (3, 8, "3"),
            (9, 14, "6"),
            (15, 20, "9"),
            (21, 26, "12"),
            (27, 32, "15"),
        ]

        for start, end, time_label in bins: # 3分毎のbinごとにデータを処理
            freezing = (
                col5.iloc[start - 1:end] # 3:8, 9:14, ...
                .astype(float)           # 数値に変換
                .mul(100 / 30)           # 30秒ごとのデータを3分ごとの平均に変換
                .mean()                  # 平均値を計算  
            )
            rows.append({
                "No": Path(file).stem,   # ファイル名を追加
                "Group": infer_group(Path(file).stem), # グループを推測して追加
                "Time": time_label,      # ラベルを追加
                "Freezing": freezing,    # 平均値を追加
            })

    return pd.DataFrame(rows)

# 既存のフォルダ
folder_Ex1 = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\Ex1"
folder_Ex2 = r"C:\Users\sryoh\Documents\Python_MSSE_analysis\Ex2"

files_Ex1 = sorted(Path(folder_Ex1).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート
files_Ex2 = sorted(Path(folder_Ex2).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート

# 3分ごとのデータ
if not files_Ex1 and not files_Ex2:
    print("指定されたフォルダに .xls ファイルはありません")
else:
    dataEx1_per3 = read_extinction_per3(files_Ex1) # Ex1の3分ごとのデータを読み込む
    dataEx2_per3 = read_extinction_per3(files_Ex2) # Ex2の3分ごとのデータを読み込む

    print(f"Ex1: {len(files_Ex1)} 件, Ex2: {len(files_Ex2)} 件") 
    print(dataEx1_per3.head()) 
    print(dataEx2_per3.head())

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but 

In [20]:
# pingouinライブラリを用いて統計検定

import pingouin as pg


# -------------------------------------------------------------------------
# 2. 対応のある二次元配置分散分析の実行 (Mauchlyの検定・Huynh-Feldt補正含む)
# -------------------------------------------------------------------------
print("=== 対応のある二次元配置分散分析 (Repeated Measures ANOVA) ===")
# pingouinのrm_anovaを使用
anova_FC = pg.mixed_anova(
    data=dataFC, 
    dv='Freezing', 
    between='Group', # 水準間要因
    within='Time',   # 水準内要因
    subject='No'     # 個体識別列
    )
print(anova_FC)
# ※ 'eps' はGreenhouse-Geisserのε。pingouinは自動で球面性を判定し、
# 必要に応じてHuynh-Feldt(HF)やGreenhouse-Geisser(GG)の調整p値を出力します。


=== 対応のある二次元配置分散分析 (Repeated Measures ANOVA) ===


ValueError: DV must be numeric.

In [ ]:

print(dataFC.columns.tolist())
print(dataFC.reset_index(drop=True))

['No', 'Group', 'Time', 'Freezing']
           No Group  Time   Freezing
0    100FCMOE   MOE     1        0.0
1    100FCMOE   MOE     2        0.0
2    100FCMOE   MOE     3        0.0
3    100FCMOE   MOE     4       47.6
4    100FCMOE   MOE     5       43.5
..        ...   ...   ...        ...
139   99FCMOE   MOE     2        0.0
140   99FCMOE   MOE     3        0.0
141   99FCMOE   MOE     4  14.566667
142   99FCMOE   MOE     5       60.8
143   99FCMOE   MOE     6  71.966667

[144 rows x 4 columns]


In [4]:
import pingouin as pg
import pandas as pd

# ① Freezing 列を明示的に数値型 (float) に変換
dataFC['Freezing'] = pd.to_numeric(dataFC['Freezing'], errors='coerce').astype(float)

# ② 欠損値（NaN）があれば除外（念のため）
dataFC = dataFC.dropna(subset=['Freezing'])

# ③ データ型を確認（Freezing が float64 になっていればOK）
print("--- 列のデータ型 ---")
print(dataFC.dtypes)
print("--------------------\n")

print("=== 混合二元配置分散分析 (Mixed ANOVA) ===")

# ④ 混合二元配置分散分析 (Mixed ANOVA) を実行
anova_FC = pg.mixed_anova(
    data=dataFC, 
    dv='Freezing', 
    between='Group', # 被験者間要因（群: SED, LIE, MOE）
    within='Time',   # 被験者内要因（測定時間: 1~6）
    subject='No'     # 個体識別列（被験者ID）
)

# ⑤ 結果の表示
print(anova_FC)

--- 列のデータ型 ---
No              str
Group           str
Time          int64
Freezing    float64
dtype: object
--------------------

=== 混合二元配置分散分析 (Mixed ANOVA) ===
        Source            SS  DF1  DF2            MS          F         p_unc  \
0        Group    113.542654    2   21     56.771327   0.149701  8.618761e-01   
1         Time  84911.381358    5  105  16982.276272  88.603689  4.591280e-36   
2  Interaction   2943.335216   10  105    294.333522   1.535662  1.369809e-01   

      p_GG_corr       np2       eps sphericity   W_spher   p_spher  
0           NaN  0.014057       NaN        NaN       NaN       NaN  
1  3.535584e-24  0.808401  0.631812      False  0.157955  0.000401  
2           NaN  0.127593       NaN        NaN       NaN       NaN  


In [5]:
# Extinction Day 1

import pingouin as pg
import pandas as pd

# ① Freezing 列を数値型に変換
dataEx1_per3['Freezing'] = pd.to_numeric(dataEx1_per3['Freezing'], errors='coerce').astype(float)
dataEx1_per3 = dataEx1_per3.dropna(subset=['Freezing'])

print("=== 消去学習 Day 1: 混合二元配置分散分析 (Mixed ANOVA) ===")

# ② 混合二元配置分散分析を実行
anova_Ex1 = pg.mixed_anova(
    data=dataEx1_per3, 
    dv='Freezing', 
    between='Group', # 被験者間要因（群）
    within='Time',   # 被験者内要因（時間: 3, 6, 9, 12, 15）
    subject='No'     # 個体識別列
)

# ③ 結果を表示
print(anova_Ex1)

=== 消去学習 Day 1: 混合二元配置分散分析 (Mixed ANOVA) ===
        Source            SS  DF1  DF2           MS          F         p_unc  \
0        Group  18192.465932    2   21  9096.232966  12.122651  3.160684e-04   
1         Time  29755.984850    4   84  7438.996212  37.024778  8.044190e-18   
2  Interaction   7601.803903    8   84   950.225488   4.729386  8.209303e-05   

      p_GG_corr       np2       eps sphericity   W_spher   p_spher  
0           NaN  0.535863       NaN        NaN       NaN       NaN  
1  2.719147e-10  0.638086  0.623494      False  0.276001  0.001163  
2           NaN  0.310543       NaN        NaN       NaN       NaN  


In [6]:
# Extinction Day 2
import pingouin as pg
import pandas as pd

# ① Freezing 列を数値型に変換
dataEx2_per3['Freezing'] = pd.to_numeric(dataEx2_per3['Freezing'], errors='coerce').astype(float)
dataEx2_per3 = dataEx2_per3.dropna(subset=['Freezing'])

print("=== 消去学習 Day 2: 混合二元配置分散分析 (Mixed ANOVA) ===")

# ② 混合二元配置分散分析を実行
anova_Ex2 = pg.mixed_anova(
    data=dataEx2_per3, 
    dv='Freezing', 
    between='Group', # 被験者間要因（群）
    within='Time',   # 被験者内要因（時間: 3, 6, 9, 12, 15）
    subject='No'     # 個体識別列
)

# ③ 結果を表示
print(anova_Ex2)

=== 消去学習 Day 2: 混合二元配置分散分析 (Mixed ANOVA) ===
        Source            SS  DF1  DF2           MS         F     p_unc  \
0        Group  19563.281243    2   21  9781.640621  6.093332  0.008188   
1         Time   1030.608677    4   84   257.652169  0.676124  0.610389   
2  Interaction   6328.662595    8   84   791.082824  2.075939  0.047159   

        np2       eps  
0  0.367216       NaN  
1  0.031192  0.786332  
2  0.165072       NaN  


In [20]:
import pingouin as pg
import pandas as pd
import numpy as np

# -------------------------------------------------------------
# 1. Shaffer の多重比較補正関数 (3群比較: [3, 1, 1] 補正)
# -------------------------------------------------------------
def shaffer_correction(pvals):
    pvals = np.asarray(pvals)
    n = len(pvals)
    order = np.argsort(pvals)
    sorted_p = pvals[order]
    
    # 3比較の場合のShaffer係数: [3, 1, 1]
    divs = [3, 1, 1] if n == 3 else [max(1, n - i) for i in range(n)]
    adj_p = np.empty(n)
    for i in range(n):
        adj_p[i] = min(1.0, sorted_p[i] * divs[i])
        if i > 0:
            adj_p[i] = max(adj_p[i], adj_p[i-1]) # 単調性の確保
            
    res_p = np.empty(n)
    res_p[order] = adj_p
    return res_p

def run_shaffer_posthoc(data, label="Extinction"):
    # 型変換
    df_data = data.copy()
    df_data['Freezing'] = pd.to_numeric(df_data['Freezing'], errors='coerce').astype(float)
    df_data = df_data.dropna(subset=['Freezing'])
    
    # Pingouinのペアワイズ検定（補正なしでt検定を実行）
    tests = pg.pairwise_tests(
        data=df_data, 
        dv='Freezing', 
        between='Group', 
        within='Time', 
        subject='No', 
        padjust='none'
    )
    
    print(f"\n{'='*75}")
    print(f"=== {label}: 各Timeにおける群間多重比較 (Shaffer method) ===")
    print(f"{'='*75}")
    
    # Time * Group の対比較（各Timeポイントでの3群比較）
    inter = tests[tests['Contrast'] == 'Time * Group'].copy()
    shaffer_list = []
    for t in inter['Time'].unique():
        sub = inter[inter['Time'] == t].copy()
        sub['p_shaffer'] = shaffer_correction(sub['p_unc'].values)
        shaffer_list.append(sub)
        
    res_time = pd.concat(shaffer_list, ignore_index=True)
    res_time['sig'] = np.select(
        [res_time['p_shaffer'] < 0.001, res_time['p_shaffer'] < 0.01, res_time['p_shaffer'] < 0.05, res_time['p_shaffer'] < 0.1],
        ['***', '**', '*', '† (p<.1)'],
        default='ns'
    )
    # Time順に並び替え
    res_time['Time_num'] = pd.to_numeric(res_time['Time'], errors='coerce')
    res_time = res_time.sort_values(by=['Time_num', 'A', 'B']).drop(columns=['Time_num'])
    
    print(res_time[['Time', 'A', 'B', 'T', 'dof', 'p_unc', 'p_shaffer', 'sig', 'hedges']].to_string(index=False))
    
    # Group 主効果全体の対比較
    print(f"\n--- {label}: 群全体の主効果に対する対比較 (Shaffer method) ---")
    grp = tests[tests['Contrast'] == 'Group'].copy()
    grp['p_shaffer'] = shaffer_correction(grp['p_unc'].values)
    grp['sig'] = np.select(
        [grp['p_shaffer'] < 0.001, grp['p_shaffer'] < 0.01, grp['p_shaffer'] < 0.05, grp['p_shaffer'] < 0.1],
        ['***', '**', '*', '† (p<.1)'],
        default='ns'
    )

    # 出力用テキストを作成
    output_text = (
        f"--- 各Timeにおける群間多重比較 (Shaffer method) ---\n"
        f"{res_time[['Time', 'A', 'B', 'T', 'dof', 'p_unc', 'p_shaffer', 'sig', 'hedges']].to_string(index=False)}\n\n"
        f"--- 群全体の主効果に対する対比較 (Shaffer method) ---\n"
        f"{grp[['A', 'B', 'T', 'dof', 'p_unc', 'p_shaffer', 'sig', 'hedges']].to_string(index=False)}"
    )

    return output_text


# -------------------------------------------------------------
# 2. 変数の指定と実行
# -------------------------------------------------------------
ex1_data = dataEx1_per3 if 'dataEx1_per3' in locals() else dataEx1per3
ex2_data = dataEx2_per3 if 'dataEx2_per3' in locals() else dataEx2per3

# 消去学習 Day 1 の多重比較
shaffer_Ex1 = run_shaffer_posthoc(ex1_data, label="消去学習 Day 1 (Ex1)")

# 消去学習 Day 2 の多重比較
shaffer_Ex2 = run_shaffer_posthoc(ex2_data, label="消去学習 Day 2 (Ex2)")

print(shaffer_Ex1)
print(shaffer_Ex2)


=== 消去学習 Day 1 (Ex1): 各Timeにおける群間多重比較 (Shaffer method) ===
Time   A   B         T  dof    p_unc  p_shaffer sig    hedges
   3 LIE MOE -0.942202 14.0 0.362065   0.362065  ns -0.445405
   3 LIE SED -2.031606 14.0 0.061624   0.184872  ns -0.960396
   3 MOE SED -0.535856 14.0 0.600468   0.600468  ns -0.253314
   6 LIE MOE -1.105025 14.0 0.287778   0.287778  ns -0.522375
   6 LIE SED -2.113265 14.0 0.053015   0.159044  ns -0.998998
   6 MOE SED -1.048872 14.0 0.312007   0.312007  ns -0.495830
   9 LIE MOE  0.581695 14.0 0.570021   0.570021  ns  0.274983
   9 LIE SED -2.516873 14.0 0.024649   0.024649   * -1.189794
   9 MOE SED -3.382133 14.0 0.004469   0.013407   * -1.598826
  12 LIE MOE  0.491193 14.0 0.630901   0.630901  ns  0.232200
  12 LIE SED -4.096660 14.0 0.001089   0.001089  ** -1.936603
  12 MOE SED -6.991819 14.0 0.000006   0.000019 *** -3.305223
  15 LIE MOE  0.018832 14.0 0.985241   0.985241  ns  0.008903
  15 LIE SED -3.851111 14.0 0.001764   0.005291  ** -1.820525
  15 MOE S

In [24]:
import pingouin as pg
import pandas as pd

# ① 1個体ごとの15分平均値を計算
sum_ex1 = dataEx1_per3.groupby(['No', 'Group'], as_index=False)['Freezing'].mean()

print("=== 15分間全体の総すくみ率に対する検定 (Day 1) ===")

# 1. 正規性の検定 (Shapiro-Wilk)
print("\n--- 1. 正規性の検定 (Shapiro-Wilk) ---")
norm_Ex1 = pg.normality(data=sum_ex1, dv='Freezing', group='Group')
print(norm_Ex1)

# 2. 等分散性の検定 (Levene)
print("\n--- 2. 等分散性の検定 (Levene) ---")
homo_Ex1 = pg.homoscedasticity(data=sum_ex1, dv='Freezing', group='Group')
print(homo_Ex1)

# 3. 一元配置分散分析 (One-way ANOVA)
print("\n--- 3. 一元配置分散分析 (One-way ANOVA) ---")
aov_Ex1 = pg.anova(data=sum_ex1, dv='Freezing', between='Group', detailed=True)
print(aov_Ex1)

# 4. Tukey HSD 多重比較
print("\n--- 4. Tukey HSD 多重比較 ---")
tukey_Ex1 = pg.pairwise_tukey(data=sum_ex1, dv='Freezing', between='Group')
print(tukey_Ex1)

=== 15分間全体の総すくみ率に対する検定 (Day 1) ===

--- 1. 正規性の検定 (Shapiro-Wilk) ---
              W      pval  normal
Group                            
MOE    0.909791  0.352595    True
SED    0.686238  0.001572   False
LIE    0.971668  0.910764    True

--- 2. 等分散性の検定 (Levene) ---
               W      pval  equal_var
levene  0.493238  0.617545       True

--- 3. 一元配置分散分析 (One-way ANOVA) ---
   Source           SS  DF           MS          F     p_unc       np2
0   Group  3638.493186   2  1819.246593  12.122651  0.000316  0.535863
1  Within  3151.470747  21   150.070036        NaN       NaN       NaN

--- 4. Tukey HSD 多重比較 ---
     A    B     mean_A     mean_B       diff        se         T   p_tukey  \
0  LIE  MOE  63.279722  63.946111  -0.666389  6.125154 -0.108795  0.993498   
1  LIE  SED  63.279722  89.725833 -26.446111  6.125154 -4.317624  0.000852   
2  MOE  SED  63.946111  89.725833 -25.779722  6.125154 -4.208829  0.001102   

     hedges  
0 -0.053782  
1 -1.776911  
2 -2.273130  


In [25]:
import pingouin as pg
import pandas as pd

# ① 1個体ごとの15分平均値を計算
sum_ex2 = dataEx2_per3.groupby(['No', 'Group'], as_index=False)['Freezing'].mean()

print("=== 15分間全体の総すくみ率に対する検定 (Day 2) ===")

# 1. 正規性の検定 (Shapiro-Wilk)
print("\n--- 1. 正規性の検定 (Shapiro-Wilk) ---")
norm_Ex2 = pg.normality(data=sum_ex2, dv='Freezing', group='Group')
print(norm_Ex2)

# 2. 等分散性の検定 (Levene)
print("\n--- 2. 等分散性の検定 (Levene) ---")
homo_Ex2 = pg.homoscedasticity(data=sum_ex2, dv='Freezing', group='Group')
print(homo_Ex2)

# 3. 一元配置分散分析 (One-way ANOVA)
print("\n--- 3. 一元配置分散分析 (One-way ANOVA) ---")
aov_Ex2 = pg.anova(data=sum_ex2, dv='Freezing', between='Group', detailed=True)
print(aov_Ex2)

# 4. Tukey HSD 多重比較
print("\n--- 4. Tukey HSD 多重比較 ---")
tukey_Ex2 = pg.pairwise_tukey(data=sum_ex2, dv='Freezing', between='Group')
print(tukey_Ex2)

=== 15分間全体の総すくみ率に対する検定 (Day 2) ===

--- 1. 正規性の検定 (Shapiro-Wilk) ---
              W      pval  normal
Group                            
MOE    0.896008  0.265855    True
SED    0.946528  0.676214    True
LIE    0.897125  0.272158    True

--- 2. 等分散性の検定 (Levene) ---
               W      pval  equal_var
levene  2.073694  0.150705       True

--- 3. 一元配置分散分析 (One-way ANOVA) ---
   Source           SS  DF           MS         F     p_unc       np2
0   Group  3912.656249   2  1956.328124  6.093332  0.008188  0.367216
1  Within  6742.270563  21   321.060503       NaN       NaN       NaN

--- 4. Tukey HSD 多重比較 ---
     A    B     mean_A     mean_B       diff        se         T   p_tukey  \
0  LIE  MOE  42.780000  35.486111   7.293889  8.959081  0.814134  0.698588   
1  LIE  SED  42.780000  65.471667 -22.691667  8.959081 -2.532812  0.048758   
2  MOE  SED  35.486111  65.471667 -29.985556  8.959081 -3.346946  0.008229   

     hedges  
0  0.412413  
1 -1.342028  
2 -1.370309  


In [12]:
# Fear Condtioningの結果をtxtファイル形式で保存

save_file_FC = Path("C:/Users/sryoh/Documents/Python_MSSE_analysis/Results/Contextual_Fear_Conditioning_Analysis.txt")

with open(save_file_FC, "w", encoding="utf-8") as fc:
    fc.write("=== Contextual fear Conditioning 統計解析結果 ===\n\n")

    fc.write(anova_FC.to_string(index=False) + "\n\n")

In [27]:
# Ex1の結果をtxtファイル形式で保存

save_file_Ex1 = Path("C:/Users/sryoh/Documents/Python_MSSE_analysis/Results/Contextual_Extinction_Day1_Analysis.txt")

with open(save_file_Ex1, "w", encoding="utf-8") as f:
    f.write("=== Contextual Fear Extinction Day 1 全体統計解析結果 ===\n\n")
    
    # 1. Mixed ANOVA
    f.write("【1. 二元配置分散分析】\n")
    f.write(anova_Ex1.to_string(index=False) + "\n\n")

    f.write("[Shafferの多重比較] \n")
    f.write(shaffer_Ex1 + "\n\n")
    
    # 2. One-way ANOVA
    f.write("【2. 15分間平均の一元配置分散分析 】\n")
    f.write("--- 正規性の検定 (Shapiro-Wilk) ---\n")
    f.write(norm_Ex1.to_string() + "\n\n")
    f.write("--- 等分散性の検定 (Levene) ---\n")
    f.write(homo_Ex1.to_string() + "\n\n")
    f.write(aov_Ex1.to_string(index=False) + "\n\n")
    
    # 3. Tukey HSD
    f.write("【3. Tukey HSD 多重比較】\n")
    f.write(tukey_Ex1.to_string(index=False) + "\n")

print(f"全解析結果を保存しました: {save_file_Ex1.resolve()}")

全解析結果を保存しました: C:\Users\sryoh\Documents\Python_MSSE_analysis\Results\Contextual_Extinction_Day1_Analysis.txt


In [ ]:
# Ex2の結果をtxtファイル形式で保存

save_file_Ex2 = Path("C:/Users/sryoh/Documents/Python_MSSE_analysis/Results/Contextual_Extinction_Day2_Analysis.txt")

with open(save_file_Ex2, "w", encoding="utf-8") as f:
    f.write("=== Contextual Fear Extinction Day 2 全体統計解析結果 ===\n\n")
    
    # 1. Mixed ANOVA
    f.write("【1. 二元配置分散分析】\n")
    f.write(anova_Ex2.to_string(index=False) + "\n\n")

    f.write("[Shafferの多重比較] \n")
    f.write(shaffer_Ex2 + "\n\n")
    
    # 2. One-way ANOVA
    f.write("【2. 15分間平均の一元配置分散分析 】\n")
    f.write("--- 正規性の検定 (Shapiro-Wilk) ---\n")
    f.write(norm_Ex2.to_string() + "\n\n")
    f.write("--- 等分散性の検定 (Levene) ---\n")
    f.write(homo_Ex2.to_string() + "\n\n")
    f.write(aov_Ex2.to_string(index=False) + "\n\n")
    
    # 3. Tukey HSD
    f.write("【3. Tukey HSD 多重比較】\n")
    f.write(tukey_Ex2.to_string(index=False) + "\n")

print(f"全解析結果を保存しました: {save_file_Ex2.resolve()}")

全解析結果を保存しました: C:\Users\sryoh\Documents\Python_MSSE_analysis\Results\Contextual_Extinction_Day2_Analysis.txt
